Groundwater | Flow Modeling Track

# Step 4: Model Implementation – MODFLOW 6

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Steps 1-3, you defined clear objectives, developed a perceptual model of the Limmat Valley aquifer, and learned MODFLOW 6 fundamentals. Now we **build the numerical model** – translating the perceptual understanding into a working FloPy simulation.

| **Core content** | ~90-110 minutes |
|:--|:--|
| **Optional: Additional details & exploration** | +5-10 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement** a groundwater flow model using FloPy and MODFLOW 6
2. **Create** unstructured (DISV) Voronoi grids with river alignment
3. **Apply** boundary conditions appropriate for the Limmat Valley
4. **Run** and verify a MODFLOW 6 simulation
5. **Interpret** model results and water balance

## Prerequisites

Before starting this notebook, you should have:
- **Completed [2_perceptual_model.ipynb](2_perceptual_model.ipynb)** – understanding of Limmat Valley hydrogeology
- **Completed [3_modflow_fundamentals.ipynb](3_modflow_fundamentals.ipynb)** – MODFLOW 6 packages and DISV concepts
- Familiarity with MODFLOW packages (NPF, CHD, RIV, WEL, RCH)
- Basic Python experience

> **Where does data go?** Files downloaded in this notebook are saved to `~/applied_groundwater_modelling_data/`. The download functions print the exact path each time.

---

## Introduction

In this notebook, we build a **numerical groundwater flow model** for the Limmat Valley aquifer using MODFLOW 6 and FloPy. The model uses an unstructured Voronoi grid (DISV) which allows flexible local refinement for your case study work.

### What is FloPy?

**FloPy** is a Python package for creating, running, and post-processing MODFLOW models — the "Python interface" to MODFLOW:

- **What it does:** Creates all the input files MODFLOW needs, runs the model, and reads results
- **Why use it:** Scriptable, reproducible, version-controlled modeling (no clicking through GUIs)
- **How it works:** You write Python code that defines your model; FloPy translates this to MODFLOW input files

```python
# Example: What FloPy code looks like (you'll build this step by step below)
import flopy
sim = flopy.mf6.MFSimulation(sim_name="example")
gwf = flopy.mf6.ModflowGwf(sim, modelname="flow")
# ... define grid, properties, boundaries ...
sim.write_simulation()
sim.run_simulation()
```

### Workflow

We follow the steps a professional groundwater modeler would take:
1. Set up the model grid with river-aligned cells (Chapters 1-2)
2. Define model geometry from DEM and aquifer thickness data (Chapter 3)
3. Assign hydraulic parameters (Chapter 4)
4. Apply boundary conditions: CHD, WEL, RIV, RCH (Chapter 5)
5. Run and verify the model (Chapters 6-7)

> **💡 Tip: Use the Outline Panel**
> This is a long notebook with many sections. To keep track of where you are:
> - **In VS Code**: Open the Outline panel (View → Open View → Outline)
> - **In JupyterHub/JupyterLab**: Open the Table of Contents panel (View → Table of Contents)

## 1 - Setup

First, we import the necessary libraries and set up the model workspace.

In [ ]:
# Setup - import required libraries
import sys
import os
import warnings
# Suppress known non-critical warnings (shapely deprecations, FloPy format notices)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*GeoSeries.*')

# Scientific computing
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# FloPy - MODFLOW 6
import flopy
from flopy.mf6 import MFSimulation, ModflowTdis, ModflowIms
from flopy.mf6 import ModflowGwf, ModflowGwfdisv
from flopy.mf6 import ModflowGwfnpf, ModflowGwfic, ModflowGwfoc
from flopy.mf6 import ModflowGwfrcha, ModflowGwfchd, ModflowGwfwel, ModflowGwfriv

# Course utilities
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from disv_grid_utils import create_grid_with_rivers, export_grid_to_geopackage
from boundary_utils import assign_idomain_from_geometry, build_lateral_inflow_wel, assign_well_cells
from river_utils import build_riv_stress_period_data
from model_io_utils import sample_raster_at_cells, format_budget_summary, load_and_interpolate_aquifer_thickness
from plot_utils import quick_model_plot
from plot_utils import plot_chd_validation

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

print(f"FloPy version: {flopy.__version__}")

In [ ]:
# Define workspace paths using os.path.join (consistent with other notebooks)
DATA_DIR = get_default_data_folder()
MODEL_WS = os.path.join(DATA_DIR, 'notebook4_model')

if not os.path.exists(MODEL_WS):
    os.makedirs(MODEL_WS)

print(f"Data directory: {DATA_DIR}")
print(f"Model workspace: {MODEL_WS}")

### 1.1 - Create MODFLOW 6 Simulation

MODFLOW 6 uses a simulation framework that can contain multiple models. We start by creating the simulation and time discretization.

In [ ]:
# Create a new MF6 simulation
sim_name = 'limmat_valley'
sim = MFSimulation(
    sim_name=sim_name,
    sim_ws=MODEL_WS,
    exe_name='mf6'  # MODFLOW 6 executable
)

# Time discretization - steady state (1 stress period)
tdis = ModflowTdis(
    sim,
    nper=1,  # Number of stress periods
    perioddata=[(1.0, 1, 1.0)]  # (perlen, nstp, tsmult)
)

print("Simulation created: steady-state, single stress period")

---

## 2 - Model Grid - DISV

We use a Voronoi-based unstructured grid (DISV = DISretization by Vertices). This grid type:
- Satisfies the control volume finite difference (CVFD) requirements
- Allows flexible local refinement (which you'll use in your case study)
- Is the modern approach for complex geometries

**River-Aligned Grid:** We incorporate river polygons as internal boundaries during grid generation. This ensures that:
- Voronoi cell edges align with river banks
- River-aquifer exchange calculations (RIV package) are more accurate
- Cells within/near rivers have finer resolution

**Recall from Notebook 3:** Voronoi cells have the property that cell centers are equidistant from all cell faces, ensuring flow directions are perpendicular to cell faces.

In [ ]:
# Load model boundary and river polygons
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

rivers_path = download_named_file(name='rivers', data_type='gis')
rivers_gdf = gpd.read_file(rivers_path)

# Filter for Sihl and Limmat rivers (main surface water bodies)
rivers_gdf = rivers_gdf[rivers_gdf['GEWAESSERNAME'].isin(['Sihl', 'Limmat'])]

print(f"Model boundary CRS: {boundary_gdf.crs}")
print(f"Boundary area: {boundary_gdf.geometry.area.sum() / 1e6:.1f} km²")
print(f"Rivers loaded: {rivers_gdf['GEWAESSERNAME'].unique().tolist()}")

In [ ]:
# Create Voronoi grid with rivers as internal boundaries
# This ensures cells align with river banks for accurate RIV package calculations

cell_size = 50  # meters - target cell size for general domain
river_cell_size = 25  # meters - finer cells in/near rivers

grid_data = create_grid_with_rivers(
    boundary_gdf=boundary_gdf,
    river_gdf=rivers_gdf,
    cell_size=cell_size,
    river_cell_size=river_cell_size,
    crs=str(boundary_gdf.crs)
)

# Extract grid components
vor = grid_data['voronoi_grid']
modelgrid = grid_data['modelgrid']
gridprops = grid_data['disv_gridprops']
ncpl = gridprops['ncpl']  # Number of cells per layer
river_cells = grid_data['river_cells']

# Expected: ~17,000-20,000 cells for 50m grid with 25m river refinement over ~10 km²
print(f"\nGrid created with {ncpl} cells")
print(f"Target cell size: {cell_size} m (general), {river_cell_size} m (rivers)")
print(f"River cells identified: {len(river_cells)}")

In [ ]:
# Visualize the grid with river cells highlighted
fig, ax = plt.subplots(figsize=(12, 10))

# Create array for river cell highlighting (0 = regular, 1 = river)
river_mask = np.zeros(ncpl)
if len(river_cells) > 0:
    river_mask[river_cells] = 1

# Use FloPy's PlotMapView for unstructured grid visualization
from flopy.plot import PlotMapView
pmv = PlotMapView(modelgrid=modelgrid, ax=ax)

# Plot grid with river cells colored
colors = pmv.plot_array(river_mask, cmap='Blues', alpha=0.6)
pmv.plot_grid(linewidth=0.3, color='gray')

# Add boundary and rivers
boundary_gdf.boundary.plot(ax=ax, color='red', linewidth=2, label='Model boundary')
rivers_gdf.boundary.plot(ax=ax, color='blue', linewidth=1.5, label='River banks')

ax.set_title(f'River-Aligned Voronoi Grid ({ncpl} cells)\n'
             f'General: ~{cell_size}m, Rivers: ~{river_cell_size}m')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

> **✏️ Quick Check**
>
> Without looking at the code, can you explain why river polygons were passed to `create_grid_with_rivers()`?

<details>
<summary>Click to check your answer</summary>

River polygons create internal boundaries that force Voronoi cell edges to align with river banks, improving accuracy of river-aquifer exchange calculations in the RIV package.
</details>

---

## 3 - Model Geometry

Now we define the vertical geometry of the model:
- **Model top**: Land surface elevation from DEM
- **Model bottom**: Computed from aquifer thickness data

In [ ]:
# Load DEM and sample at cell centers
dem_path = download_named_file(name='dem_hres', data_type='gis')

# Sample DEM at cell centers using FloPy's Raster utility
model_top = sample_raster_at_cells(dem_path, modelgrid)

print(f"Model top elevation range: {model_top.min():.1f} to {model_top.max():.1f} m a.s.l.")

In [ ]:
# Aquifer thickness interpolation
# The aquifer thickness varies spatially based on geological survey data.
# Primary source: thickness contour lines (GS_GW_MAECHTIGKEIT_L layer)
# Additional data: shallow aquifer zone polygons (GS_GW_LEITER_F layer) are
# blended in by default to improve estimates at domain margins.
# Interpolation uses two stages:
# 1. Linear interpolation for smooth interior surfaces
# 2. Nearest-neighbor gap filling for areas outside the convex hull

# Download thickness data (same file as groundwater map)
gw_map_path = download_named_file(name='groundwater_map_norm', data_type='gis')

# Minimum aquifer thickness constraint (modeling, not physical observation):
# Where interpolation data is sparse (e.g. at domain margins), thickness values
# may be unrealistically small. We impose a 10 m floor to avoid numerical issues
# (very thin cells → extreme head gradients). The true aquifer may be thinner
# locally, but this has negligible impact on regional flow patterns.
min_aquifer_thickness = 10.0  # meters

# Interpolate thickness at grid cell centers
# The function clips internally to [min_thickness, max_thickness]
aquifer_thickness = load_and_interpolate_aquifer_thickness(
    gw_map_path=gw_map_path,
    modelgrid=modelgrid,
    boundary_gdf=boundary_gdf,
    min_thickness=min_aquifer_thickness
)

# Compute model bottom elevation
model_bottom = model_top - aquifer_thickness

print(f"\nModel geometry summary:")
print(f"  Model top: {model_top.min():.1f} to {model_top.max():.1f} m a.s.l.")
print(f"  Thickness: {aquifer_thickness.min():.1f} to {aquifer_thickness.max():.1f} m")
print(f"  Minimum thickness constraint: {min_aquifer_thickness} m")
print(f"  Model bottom: {model_bottom.min():.1f} to {model_bottom.max():.1f} m a.s.l.")

In [ ]:
# Visualize model geometry
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

quick_model_plot(modelgrid, model_top, boundary_gdf=boundary_gdf,
                 title='Model Top (Land Surface)', cmap='terrain',
                 colorbar_label='Elevation (m a.s.l.)', ax=axes[0])

quick_model_plot(modelgrid, aquifer_thickness, boundary_gdf=boundary_gdf,
                 title='Aquifer Thickness', cmap='Blues',
                 colorbar_label='Thickness (m)', ax=axes[1])

plt.tight_layout()
plt.show()

### 3.1 - ✏️ Checkpoint 1: Aquifer Volume

Using the model domain area (printed in the grid section above) and the average aquifer thickness you just computed, **calculate the total aquifer volume**.

**Hint:** $V = A \times \bar{b}$. Convert your answer to million m³.

In [ ]:
# Checkpoint 1: Calculate aquifer volume from domain area and average thickness
# Use the average thickness printed in the model geometry summary above
check_task_with_solution('task04_checkpoint_1')

---

## 4 - Hydraulic Parameters

The Limmat Valley aquifer consists of Quaternary alluvial and glacial deposits with hydraulic conductivity (K) typically ranging from 10-40 m/day. For this **base model**, we use a **uniform K value** as a starting point.

| Parameter | Value | Justification |
|-----------|-------|---------------|
| K (uniform) | 20 m/day | Mid-range for sandy gravels, from pumping tests |

> **📚 Theory: Why Uniform K?**
>
> We use uniform K for this initial model to keep the implementation focused. In practice, K varies spatially due to:
> - Different sediment facies (sandy gravels upstream, finer alluvium downstream)
> - Depositional history (glacial vs. fluvial processes)
>
> **Future refinements:**
> - **Calibration (Notebook 5)** will optimize K against observed heads
> - **Advanced models** could use stochastic K fields derived from Kriging of borehole data

> **⚠️ Uncertainty: Hydraulic Conductivity**
>
> The uniform K = 20 m/day is a starting estimate. Pumping test data in the Limmat Valley show local values ranging from **8 m/day** (fine alluvium, downstream) to **40 m/day** (coarse gravels, Hardhof area). This factor-of-4 uncertainty is the primary motivation for calibration in Notebook 5.
>
> Note: Notebook 2 cited a literature value of ~864 m/day for Limmat gravels. That value reflects laboratory-scale measurements on clean, well-sorted gravel samples — not field-scale aquifer K, which integrates heterogeneity, partial cementation, and finer interbeds. Our 20 m/day starting estimate reflects pumping test data (8-40 m/day), which better represents the effective aquifer conductivity.

### 4.1 - ✏️ Exercise: Transmissivity Calculation

Transmissivity ($T$) describes the rate at which water is transmitted through a unit width of aquifer:

$$T = K \times b$$

**Calculate** the transmissivity for the Limmat Valley aquifer using:
- $K$ = 20 m/day (uniform base model value)
- $\bar{b}$ = average aquifer thickness from the model geometry summary (Section 3)

<details>
<summary>Click to check your answer</summary>

Using the average thickness from your model (check the geometry summary above, typically ~12 m):

$$T = 20 \text{ m/day} \times \bar{b} \approx 240 \text{ m}^2\text{/day}$$

For context:
- This is equivalent to $T$ ≈ 2.8 × 10⁻³ m²/s
- Literature values for the Limmat Valley range from 80-800 m²/day depending on location
- Our estimate falls in the lower-middle of this range (with uniform K and the model's average thickness)

**Why does transmissivity matter?** It determines how quickly the aquifer can deliver water to wells and boundaries. Higher $T$ means faster response to pumping and more efficient lateral flow.
</details>

In [ ]:
# Define uniform hydraulic conductivity for the base model
k_uniform = 20.0  # m/day - mid-range for sandy gravels

print(f"Hydraulic conductivity: {k_uniform} m/day (uniform)")
print("This value will be refined during calibration (Notebook 5)")

In [ ]:
# Create uniform K array for all cells
k_array = np.full(ncpl, k_uniform)

print(f"K array shape: {k_array.shape}")
print(f"K value: {k_array[0]:.1f} m/day (uniform for base model)")

---

## 5 - Boundary Conditions

We define the boundary conditions that control water flow:

1. **IDOMAIN**: Active/inactive cells
2. **RIV**: River-aquifer interaction (Sihl & Limmat)
3. **CHD**: Constant head at outflow (west) boundary
4. **WEL**: Lateral inflow from valley margins + pumping extraction
5. **RCH**: Areal recharge from precipitation

The east boundary is impermeable bedrock (no-flow, default).

### Perceptual Model → MODFLOW Translation

The following table maps the conceptual understanding from Notebook 2 to MODFLOW 6 package inputs:

| Perceptual Model Component | Estimated Value | MODFLOW Package | Parameter |
|---------------------------|-----------------|-----------------|-----------|
| River leakage coefficient | 1.3-3.5 × 10⁻⁶ 1/s | **RIV** | conductance (via K_riverbed) |
| Lateral hillslope inflow | ~3,300 + 4,500 m³/day | **WEL** | Q (positive = injection) |
| Medium pumping concessions | ~1,640 m³/day | **WEL** | Q (negative = extraction) |
| Net recharge | ~110 mm/year | **RCH** | 3.0×10⁻⁴ m/day |
| Western outflow | ~2,000-20,000 m³/day | **CHD** | head = 390 m a.s.l. |
| Eastern boundary | Impermeable bedrock | (No-flow) | Default in MODFLOW |

### 5.1 - Active Domain (IDOMAIN)

In [ ]:
# Create IDOMAIN - all cells within boundary are active
idomain = assign_idomain_from_geometry(boundary_gdf, modelgrid, nlay=1)
n_active = np.sum(idomain > 0)
print(f"Active cells: {n_active} of {ncpl} ({100*n_active/ncpl:.1f}%)")

# Validation checks
assert idomain.shape == (1, ncpl), f"IDOMAIN shape mismatch: expected (1, {ncpl}), got {idomain.shape}"

thickness = model_top - model_bottom
if np.any(thickness[idomain[0] > 0] <= 0):
    n_bad = np.sum(thickness[idomain[0] > 0] <= 0)
    print(f"  Warning: {n_bad} active cells have zero or negative thickness")
else:
    print(f"  Validation passed: all active cells have positive thickness")

### 5.2 - River Boundary (RIV)

The Sihl and Limmat rivers exchange water with the aquifer. The RIV package calculates flux based on the head difference between river stage and aquifer head.

> **📚 Theory: RIV Conductance**
>
> Conductance controls the exchange rate. Since riverbed hydraulic conductivity ($K_{riverbed}$) and clogging layer thickness ($d$) are hard to measure independently, we use the **leakage coefficient** $\lambda = K_{riverbed} / d$ (from Doppler et al., 2007). This simplifies conductance to:
> $$C = \lambda \times W \times L$$
> where $W$ = river width and $L$ = reach length (approximated as $\sqrt{A_{cell}}$).
>
> **Note:** The $\lambda$ values from Doppler et al. are reported in 1/s; the code converts to 1/day ($\times$ 86400) before computing conductance in m²/day.

**River-specific parameters:**

| River | Width | Mean Depth | Leakage Coeff ($\lambda$) | Source |
|-------|-------|------------|---------------------------|--------|
| Sihl | 15 m | 0.4 m | 1.3×10⁻⁶ 1/s | Doppler et al. (2007) |
| Limmat | 30 m | 1.0 m | 3.5×10⁻⁶ 1/s | Doppler et al. (2007), geometric mean |

**River stage and riverbed bottom (rbot):** We compute the riverbed bottom as rbot = DEM − river_depth − 0.5 m (where 0.5 m accounts for river incision below the DEM surface), then homogenize rbot across cross-sections so that all cells in a transect share the same minimum rbot. Stage is then derived as stage = rbot + river_depth. The deeper Limmat therefore has a lower rbot (and thus a lower disconnection threshold) than the shallower Sihl.

> **⚠️ Uncertainty:** The Limmat leakage coefficient is the geometric mean of a range spanning two orders of magnitude (3.5×10⁻⁷ to 3.5×10⁻⁵ 1/s; Doppler et al., 2007). River stages are approximated from the DEM, not gauge data. These parameters will be refined during calibration.

In [ ]:
# Build RIV stress period data using the utility function
# This handles river classification, rbot homogenization, and conductance calculation

if len(river_cells) > 0:
    riv_spd, riv_cell_df = build_riv_stress_period_data(
        river_cells=river_cells,
        modelgrid=modelgrid,
        model_top=model_top,
        model_bottom=model_bottom,
        rivers_gdf=rivers_gdf,
        idomain=idomain,
        bin_width=100.0  # meters, for cross-section binning
    )
    
    # Summary statistics
    n_sihl = (riv_cell_df['river'] == 'Sihl').sum()
    n_limmat = (riv_cell_df['river'] == 'Limmat').sum()
    n_unknown = (riv_cell_df['river'] == 'Unknown').sum()
    
    # Homogenization effect
    rbot_diff = riv_cell_df['raw_rbot'] - riv_cell_df['homogenized_rbot']
    
    print(f"RIV package summary:")
    print(f"  Total river cells: {len(riv_spd)}")
    print(f"    Sihl: {n_sihl} cells")
    print(f"    Limmat: {n_limmat} cells")
    if n_unknown > 0:
        # "Unknown" cells occur at river confluences or polygon gaps where a cell
        # center doesn't fall inside either river polygon. They are assigned Limmat
        # parameters (the larger river) as a conservative default.
        print(f"    Unassigned: {n_unknown} cells (at confluences/polygon gaps, using Limmat parameters)")
    print(f"\n  Stage range: {riv_cell_df['stage'].min():.1f} to {riv_cell_df['stage'].max():.1f} m a.s.l.")
    print(f"  Conductance range: {riv_cell_df['conductance'].min():.1f} to {riv_cell_df['conductance'].max():.1f} m²/day")
    print(f"\n  Rbot homogenization:")
    print(f"    Max adjustment: {rbot_diff.max():.2f} m")
    print(f"    Mean adjustment: {rbot_diff.mean():.2f} m")
    print(f"    Cells adjusted: {(rbot_diff > 0.01).sum()} of {len(riv_cell_df)}")
else:
    riv_spd = []
    riv_cell_df = pd.DataFrame()
    print("No river cells identified - RIV package will not be created")

### Optional: Rbot Homogenization Visualization

The following plot shows the effect of cross-sectional rbot homogenization on river cells. This is a technical detail — you can skip to [Section 5.3](#53---Outflow-Boundary-CHD) without loss of continuity.

In [ ]:
# Visualize river boundary: rbot homogenization effect
if len(riv_spd) > 0 and len(riv_cell_df) > 0:
    # Get cell coordinates
    xc = modelgrid.xcellcenters.flatten() if hasattr(modelgrid.xcellcenters, 'flatten') else modelgrid.xcellcenters
    yc = modelgrid.ycellcenters.flatten() if hasattr(modelgrid.ycellcenters, 'flatten') else modelgrid.ycellcenters
    
    # Calculate rbot adjustment for color mapping
    rbot_adjustment = riv_cell_df['raw_rbot'] - riv_cell_df['homogenized_rbot']
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot river cells colored by rbot adjustment
    scatter = ax.scatter(
        riv_cell_df['x'], riv_cell_df['y'], 
        c=rbot_adjustment, cmap='RdYlBu_r', 
        s=10, alpha=0.8, 
        vmin=0, vmax=max(0.5, rbot_adjustment.max())
    )
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Rbot adjustment (m)', fontsize=10)
    
    # Add boundary
    boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5)
    
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(f'River Bottom Homogenization Effect\n(Blue = no change, Red = lowered to match cross-section minimum)')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    
    # Quick stats
    print(f"\nSummary: {(rbot_adjustment > 0.01).sum()} cells had rbot lowered (max: {rbot_adjustment.max():.2f}m)")

### 5.3 - Outflow Boundary (CHD)

The western boundary represents the valley outlet where groundwater leaves the model domain. We implement this as a Constant Head (CHD) boundary condition.

| Boundary | Location | Package | Description |
|----------|----------|---------|-------------|
| **Outflow** | West | CHD | Fixed head at ~390 m a.s.l. (valley outlet) |

The CHD head value is estimated from groundwater isoline maps at the valley outlet.

> **⚠️ Uncertainty: Outflow Head**
>
> The CHD head of 390 m a.s.l. is estimated from groundwater isoline maps. The actual head at the valley outlet may vary by ±2-3 m depending on season and river stage. In calibration, this value could be adjusted if systematic head bias is observed near the western boundary.

In [ ]:
# CHD at outflow boundary (west)
# Load boundary segments to identify outflow cells
segments_path = download_named_file(name='model_boundary_segments', data_type='gis')
boundary_segments = gpd.read_file(segments_path)

# Create cell center points for spatial join
xc = modelgrid.xcellcenters.flatten() if hasattr(modelgrid.xcellcenters, 'flatten') else modelgrid.xcellcenters
yc = modelgrid.ycellcenters.flatten() if hasattr(modelgrid.ycellcenters, 'flatten') else modelgrid.ycellcenters

cell_points = gpd.GeoDataFrame(
    {'cell_id': range(ncpl)},
    geometry=[Point(x, y) for x, y in zip(xc, yc)],
    crs=boundary_gdf.crs
)

# Get outflow boundary (west) and buffer slightly to capture nearby cells
outflow_line = boundary_segments[boundary_segments['desc'] == 'outflow'].geometry.values[0]
outflow_buffer = outflow_line.buffer(75)  # 75m buffer to capture cells near boundary

# Find cells within buffer of outflow boundary
outflow_cells = cell_points[cell_points.geometry.within(outflow_buffer)]
outflow_cell_ids = outflow_cells['cell_id'].tolist()

# Filter to only active cells
outflow_cell_ids = [i for i in outflow_cell_ids if idomain[0, i] > 0]

# CHD head at outflow (downstream) - from groundwater isoline map
chd_head_outflow = 390.0  # m a.s.l.

# Validation: ensure CHD head is above aquifer bottom
west_bottom = model_bottom[outflow_cell_ids].min()
if chd_head_outflow <= west_bottom:
    raise ValueError(f"CHD head {chd_head_outflow} m is below aquifer bottom {west_bottom:.1f} m!")

# Verify heads are below land surface
outflow_min_top = model_top[outflow_cell_ids].min()
if chd_head_outflow >= outflow_min_top:
    chd_head_outflow = outflow_min_top - 1.0
    print(f"Warning: Adjusted CHD head to {chd_head_outflow:.1f} m (below land surface)")

# Create CHD stress period data
chd_spd = [[(0, i), chd_head_outflow] for i in outflow_cell_ids]

print(f"CHD boundary (outflow/west):")
print(f"  Number of cells: {len(chd_spd)}")
print(f"  Head value: {chd_head_outflow:.1f} m a.s.l.")
print(f"  Aquifer bottom at CHD: {west_bottom:.1f} m (head is {chd_head_outflow - west_bottom:.1f} m above)")

In [ ]:
# Validation figure: verify CHD cells are correctly placed at the western boundary
fig, axes = plot_chd_validation(
    chd_spd=chd_spd,
    modelgrid=modelgrid,
    model_top=model_top,
    model_bottom=model_bottom,
    idomain=idomain,
    boundary_gdf=boundary_gdf,
    riv_spd=riv_spd,
    buffer_m=100,
    title="CHD Boundary Validation - Outflow (West)"
)
plt.show()

### 5.4 - Lateral Inflow (WEL)

Groundwater enters the Limmat Valley from the north and south valley margins. We use the **water balance approach** from Notebook 2:

| Boundary | Catchment Area | Effective Recharge | Estimated Inflow |
|----------|---------------|-------------------|------------------|
| North | ~11 km² | 110 mm/year (10%) | ~3,300 m³/day |
| South | ~15 km² | 110 mm/year (10%) | ~4,500 m³/day |

**Implementation:** Identify cells along each boundary edge and distribute the total inflow evenly among them.

In [ ]:
# Build lateral inflow WEL package using the utility function
# Boundary segments already loaded above (CHD section, cell 32)

# Build WEL stress period data
wel_spd, north_cell_ids, south_cell_ids = build_lateral_inflow_wel(
    boundary_segments=boundary_segments,
    modelgrid=modelgrid,
    idomain=idomain,
    buffer_distance=150.0,  # meters
    x_bin_width=50.0  # meters, for edge detection
)

# Calculate per-cell rates for display
Q_north_total = 11_000_000 * 0.11 / 365.25  # m³/day
Q_south_total = 15_000_000 * 0.11 / 365.25  # m³/day
q_per_cell_north = Q_north_total / len(north_cell_ids) if north_cell_ids else 0
q_per_cell_south = Q_south_total / len(south_cell_ids) if south_cell_ids else 0

print(f"Lateral inflow (WEL) summary:")
print(f"  North boundary: {len(north_cell_ids)} cells, Q per cell: {q_per_cell_north:.1f} m³/day")
print(f"  South boundary: {len(south_cell_ids)} cells, Q per cell: {q_per_cell_south:.1f} m³/day")
print(f"  Total inflow: {sum([w[1] for w in wel_spd]):.0f} m³/day")

In [ ]:
from map_utils import display_wel_map
display(display_wel_map(modelgrid, north_cell_ids, south_cell_ids, q_per_cell_north, q_per_cell_south))

### 5.5 - Areal Recharge (RCH)

Net areal recharge represents the water that infiltrates through the land surface and reaches the water table. In the perceptual model (Notebook 2), we estimated the net recharge for the Limmat Valley to be approximately **110 mm/year**.

This value accounts for:
- Precipitation (~1100 mm/year)
- Evapotranspiration losses
- Surface runoff
- Urbanization effects (reduced infiltration in built-up areas)

The net recharge of ~10% of annual precipitation is typical for urbanized alluvial valleys in temperate climates.

> **⚠️ Uncertainty: Recharge Rate**
>
> The 110 mm/year value represents ~10% of annual precipitation. Actual recharge varies spatially
> (higher in parks/green areas, lower in paved urban zones) and temporally (seasonal). For the
> lateral hillslope inflow, the same 110 mm/yr rate is applied to the catchment areas, though
> hillslope recharge may differ from valley-floor recharge. These simplifications will be assessed
> during calibration.

In [ ]:
# Recharge calculation
# Net recharge rate from Notebook 2 perceptual model analysis
# Value for Limmat Valley: ~110 mm/year (10% of precipitation)

annual_recharge_mm = 110  # mm/year (net recharge from perceptual model)
recharge_m_per_day = annual_recharge_mm / 1000 / 365.25  # Convert to m/day

print(f"Annual net recharge: {annual_recharge_mm} mm/year")
print(f"  (~{annual_recharge_mm/1100*100:.0f}% of annual precipitation)")
print(f"Daily recharge rate: {recharge_m_per_day:.2e} m/day")

# Apply uniform recharge to all active cells
recharge_array = np.where(idomain[0] > 0, recharge_m_per_day, 0.0)

# Total recharge flux over active domain
print(f"Estimated total recharge: {recharge_m_per_day * boundary_gdf.geometry.area.sum():.0f} m³/day")

### 5.6 - ✏️ Checkpoint 2: Areal Recharge Flux

**Verify** that you can reproduce the total recharge flux printed above using: rate (3.0 × 10⁻⁴ m/day) × area (~10.4 km²).

In [ ]:
# Check your recharge flux
# Expected: ~2,700-3,500 m³/day (based on ~10 km² × 110 mm/year)
# This should match your perceptual model estimates from Notebook 2
check_task_with_solution('task04_checkpoint_2')

### 5.7 - Groundwater Pumping (WEL)

Medium-sized groundwater concessions (~600,000 m³/year at 40% utilization, see Notebook 2) are included as consumptive extraction. Hardhof (MAR system, net effect minimal) and thermal use wells (non-consumptive) are excluded.

> **💡 Preview: Case Study Work**
>
> In your project (Notebook 8), you'll add thermal doublet wells (extraction + injection) and investigate capture zones and thermal plume propagation.

In [ ]:
# Load groundwater concessions and identify pumping wells
wells_path = download_named_file(name='wells', data_type='gis')
wells_all = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')

# Clip to model domain
wells_in_domain = gpd.clip(wells_all, boundary_gdf)

# Filter: active, medium-capacity (300-3000 L/min), consumptive use
# See Notebook 2 (Section on concession analysis) for detailed justification
is_active = ~wells_in_domain['BESCHREIBUNG'].str.contains(
    'aufgehoben|ungenutzt', case=False, na=False)
is_medium = wells_in_domain['BESCHREIBUNG'].str.contains('300', na=False)
is_consumptive = ~wells_in_domain['NUTZART'].isin(['WPG'])

pumping_wells = wells_in_domain[is_active & is_medium & is_consumptive].copy()

# Assign extraction rates
# From Notebook 2: ~600,000 m³/year total at 40% of concessioned capacity
total_pumping_m3_year = 600_000
total_pumping_m3_day = total_pumping_m3_year / 365.25

if len(pumping_wells) > 0:
    rate_per_well = -total_pumping_m3_day / len(pumping_wells)  # Negative = extraction
    pumping_wells['rate'] = rate_per_well
    pumping_spd = assign_well_cells(pumping_wells, modelgrid)

    print(f"Pumping wells: {len(pumping_wells)}")
    print(f"  Total extraction: {total_pumping_m3_day:.0f} m³/day ({total_pumping_m3_year:,.0f} m³/year)")
    print(f"  Rate per well: {rate_per_well:.1f} m³/day")
    for _, w in pumping_wells.iterrows():
        name = w.get('FASSBEZ', w.get('GWR_ID', 'Unknown'))
        print(f"    - {name} ({w['NUTZART']})")
else:
    pumping_spd = []
    print("No medium-capacity consumptive wells found in model domain")

---

All boundary conditions are now defined. We can assemble the MODFLOW 6 packages and run the simulation. The model will solve for steady-state heads that balance all fluxes.

## 6 - Build and Run Model

In [ ]:
# Create the groundwater flow model with Newton-Raphson formulation
# Newton is strongly recommended for robust unconfined flow simulation
# (avoids dry-cell rewetting issues)
gwf = ModflowGwf(sim, modelname=sim_name, save_flows=True,
                 newtonoptions='NEWTON UNDER_RELAXATION')

# DISV grid
disv = ModflowGwfdisv(gwf, nlay=1, ncpl=ncpl,
    nvert=gridprops['nvert'], vertices=gridprops['vertices'],
    cell2d=gridprops['cell2d'], top=model_top, botm=[model_bottom],
    idomain=idomain)

# Hydraulic properties (NPF) and initial conditions (IC)
npf = ModflowGwfnpf(gwf, icelltype=1, k=k_array, save_flows=True)

# Initial heads: 2m below land surface (reasonable starting point for Newton solver;
# steady-state solution converges regardless of initial condition)
initial_heads = np.maximum(model_top - 2.0, model_bottom + 0.5)
ic = ModflowGwfic(gwf, strt=initial_heads)

print(f"GWF model: {ncpl} DISV cells, K = {k_uniform} m/day, Newton-Raphson solver")
print(f"Initial heads: {initial_heads.min():.1f} to {initial_heads.max():.1f} m a.s.l.")

In [ ]:
# Boundary condition packages
chd = ModflowGwfchd(gwf, stress_period_data=chd_spd)
rcha = ModflowGwfrcha(gwf, recharge=recharge_array)

all_wel_spd = list(wel_spd) if len(wel_spd) > 0 else []
if len(pumping_spd) > 0:
    all_wel_spd.extend(pumping_spd)
if len(all_wel_spd) > 0:
    wel = ModflowGwfwel(gwf, stress_period_data=all_wel_spd)

if len(riv_spd) > 0:
    riv = ModflowGwfriv(gwf, stress_period_data=riv_spd)

# Summary
wel_inflow = sum(w[1] for w in all_wel_spd if w[1] > 0)
wel_outflow = sum(w[1] for w in all_wel_spd if w[1] < 0)
print(f"CHD: {len(chd_spd)} cells at {chd_head_outflow:.0f} m a.s.l.")
print(f"RCH: {recharge_m_per_day:.2e} m/day")
print(f"WEL: {len(all_wel_spd)} cells (inflow: {wel_inflow:.0f}, pumping: {abs(wel_outflow):.0f} m³/day)")
print(f"RIV: {len(riv_spd)} river cells")

> **🤔 Reflection:** Before running the model, consider: What drives water in (RCH, WEL, RIV)? What removes it (CHD, RIV)? The solver must find heads where all fluxes balance to zero — the steady-state solution.

In [ ]:
# Solver: Newton-Raphson with under-relaxation for robust unconfined flow
# DBD (Delta-Bar-Delta) is an adaptive under-relaxation scheme that prevents overshooting during iteration
# Convergence tolerance: 1 cm (appropriate for regional models with >20m head range)
ims = ModflowIms(
    sim, complexity='MODERATE',
    outer_maximum=500, inner_maximum=100,
    outer_dvclose=1e-2, inner_dvclose=1e-4,
    linear_acceleration='BICGSTAB',
    under_relaxation='DBD',
    under_relaxation_gamma=0.1, under_relaxation_theta=0.85,
    backtracking_number=5,
)
sim.register_ims_package(ims, [gwf.name])

# Output control
oc = ModflowGwfoc(gwf,
    head_filerecord=f'{sim_name}.hds',
    budget_filerecord=f'{sim_name}.cbc',
    saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')])

# Write and run
sim.write_simulation()
print("Running MODFLOW 6 simulation...")
success, buff = sim.run_simulation(silent=True)
print("Simulation completed successfully!" if success else
      f"Simulation failed. Check: {os.path.join(MODEL_WS, sim_name + '.lst')}")

---

The model ran successfully - but what do the results tell us? In this section, we examine the simulated heads and water balance to verify the model is behaving as expected.

## 7 - Initial Results

This is an **uncalibrated** model with initial parameter estimates.

In [ ]:
if success:
    # Read head file
    head_file = flopy.utils.HeadFile(os.path.join(MODEL_WS, f'{sim_name}.hds'))
    heads = head_file.get_data()
    head_layer1 = heads[0].flatten()
    
    # Head validation
    # MODFLOW uses special values for inactive/dry cells:
    # - HDRY (typically 1e30): Cell went dry during simulation
    # - HNOFLO (typically 1e30): Inactive cell (idomain <= 0)
    
    hdry_threshold = 1e10  # Values with |head| above this indicate dry/inactive cells
    
    # Identify problematic cells
    inactive_mask = idomain[0].flatten() <= 0
    dry_mask = (np.abs(head_layer1) > hdry_threshold) & ~inactive_mask
    valid_mask = ~inactive_mask & ~dry_mask
    
    n_inactive = inactive_mask.sum()
    n_dry = dry_mask.sum()
    n_valid = valid_mask.sum()
    
    print("Head validation:")
    print(f"  Total cells: {ncpl}")
    print(f"  Inactive cells (idomain <= 0): {n_inactive}")
    print(f"  Dry cells (|head| > {hdry_threshold:.0e}): {n_dry}")
    print(f"  Valid cells with computed heads: {n_valid}")
    
    if n_dry > 0:
        print(f"\n  ⚠️ WARNING: {n_dry} cells went dry during simulation!")
        print(f"     This may indicate:")
        print(f"     - Aquifer bottom too high relative to boundary conditions")
        print(f"     - Insufficient recharge or inflow")
        print(f"     - Excessive pumping or river leakage")
    else:
        print(f"\n  Note: The Newton-Raphson formulation (enabled above) handles dry cells")
        print(f"  by smoothly reducing transmissivity to near-zero rather than abruptly")
        print(f"  deactivating cells. This is why you may see zero dry cells even with")
        print(f"  challenging parameters.")
    
    # Report statistics for valid cells only
    if n_valid > 0:
        valid_heads = head_layer1[valid_mask]
        print(f"\nSimulated head range (valid cells):")
        print(f"  Min: {valid_heads.min():.1f} m a.s.l.")
        print(f"  Max: {valid_heads.max():.1f} m a.s.l.")
        print(f"  Mean: {valid_heads.mean():.1f} m a.s.l.")
    else:
        print("\n  ERROR: No valid head values computed!")
else:
    print("Cannot load results - simulation did not complete.")

In [ ]:
if success:
    # Create masked array for plotting (mask dry/inactive cells)
    head_plot = np.ma.masked_where(
        (np.abs(head_layer1) > hdry_threshold) | (idomain[0].flatten() <= 0),
        head_layer1
    )
    
    fig, ax = plt.subplots(figsize=(12, 10))
    quick_model_plot(
        modelgrid, head_plot, boundary_gdf=boundary_gdf,
        title='Simulated Groundwater Heads (Uncalibrated)',
        cmap='Blues', colorbar_label='Head (m a.s.l.)', ax=ax
    )
    plt.tight_layout()
    plt.show()

> **🤔 Reflection: Head Map Interpretation**
>
> Look at the simulated head map above:
> - Where are the highest heads? Does the gradient direction match the SE-to-W flow from your perceptual model (Notebook 2)?
> - Can you identify the influence of the river (lower heads along the river corridor)?

### 7.1 - Water Balance

For a steady-state model, inflows and outflows should balance (error < 0.1%).

In [ ]:
# Read and display budget components
if success:
    budget_df = format_budget_summary(gwf, sim)
    if budget_df is not None:
        print("Water Balance Summary:")
        print(budget_df.to_string())
    else:
        print("Budget summary not available.")

### 7.2 - Interpreting the Water Balance

<details>
<summary><b>Expected ranges and interpretation guidance (click to reveal)</b></summary>

For a well-functioning Limmat Valley model:
- **RIV dominates**: 50-70% of total inflow (river is the primary source)
- **CHD outflow**: 2,000-20,000 m³/day (matches Darcy estimate from Nb2)
- **Total imbalance**: < 0.1% (excellent convergence expected)
- **RCH contribution**: ~3,000 m³/day (consistent with 110 mm/year over 10 km²)
- **WEL net**: ~6,200 m³/day (lateral inflow ~7,800 minus pumping ~1,640 m³/day)

**Questions to consider:**
1. Which flux component dominates? Is this consistent with your perceptual model?
2. The RIV package shows both inflow and outflow — what does this indicate about river-aquifer interaction?
3. How does the CHD outflow compare to your Darcy-based estimate from Notebook 2?

If your values differ significantly, check boundary condition parameters (especially river conductance), model convergence, and grid resolution near boundaries.
</details>

> **⚠️ Note:** These are uncalibrated results — do not use for predictions until after calibration (Notebook 5).

### 7.3 - ✏️ Checkpoint 3: Model Convergence

In [ ]:
# Check your water balance error
# Expected: < 0.1% for a well-converged steady-state model
# Errors > 1% indicate convergence problems - check solver settings
check_task_with_solution('task04_checkpoint_3')

### 7.4 - ✏️ Checkpoint 4: River-Aquifer Interaction

In [ ]:
create_multiple_choice('task04_checkpoint_4')

### 7.5 - Sensitivity Exercise: How Does K Affect the Water Balance?

Hydraulic conductivity (K) controls how easily water flows through the aquifer. In a steady-state model, K determines the **equilibrium head distribution** — the balance point where inflows (recharge, lateral inflow, river leakage) equal outflows (CHD, river leakage).

> **✏️ Exercise: Explore the effect of K**
>
> In the code cell below, change `k_test` to different values and rerun to see how the model responds. Try values from 0.5 to 500 m/day. For each value, observe:
> 1. Does the model converge?
> 2. How many cells go dry?
> 3. Are simulated heads physically reasonable (above aquifer bottom, below land surface)?

In [ ]:
# ============================================================
# SENSITIVITY EXERCISE: Change k_test and rerun this cell
# You only need to modify the value on the next line.
# ============================================================
k_test = 20.0  # <-- CHANGE THIS VALUE (try: 0.5, 5, 20, 50, 200, 500)

# --- Rebuild model with test K value ---
# Create a fresh simulation to avoid package conflicts
sim_test = MFSimulation(sim_name='k_test', sim_ws=os.path.join(MODEL_WS, 'k_sensitivity'), exe_name='mf6')
ModflowTdis(sim_test, nper=1, perioddata=[(1.0, 1, 1.0)])

gwf_test = ModflowGwf(sim_test, modelname='k_test', save_flows=True,
                       newtonoptions='NEWTON UNDER_RELAXATION')

# Reuse grid, geometry, and boundary conditions from the base model
ModflowGwfdisv(gwf_test, nlay=1, ncpl=ncpl, nvert=gridprops['nvert'],
               vertices=gridprops['vertices'], cell2d=gridprops['cell2d'],
               top=model_top, botm=[model_bottom], idomain=idomain)

# Apply the test K value
ModflowGwfnpf(gwf_test, icelltype=1, k=np.full(ncpl, k_test), save_flows=True)
ModflowGwfic(gwf_test, strt=initial_heads)
ModflowGwfchd(gwf_test, stress_period_data=chd_spd)
ModflowGwfrcha(gwf_test, recharge=recharge_array)

# Include both lateral inflow and pumping wells
test_wel_spd = list(wel_spd) + (list(pumping_spd) if len(pumping_spd) > 0 else [])
if len(test_wel_spd) > 0:
    ModflowGwfwel(gwf_test, stress_period_data=test_wel_spd)
if len(riv_spd) > 0:
    ModflowGwfriv(gwf_test, stress_period_data=riv_spd)

ims_test = ModflowIms(sim_test, complexity='MODERATE', outer_maximum=500,
                       inner_maximum=100, outer_dvclose=1e-2, inner_dvclose=1e-4,
                       linear_acceleration='BICGSTAB', under_relaxation='DBD',
                       under_relaxation_gamma=0.1, under_relaxation_theta=0.85,
                       backtracking_number=5)
sim_test.register_ims_package(ims_test, [gwf_test.name])
ModflowGwfoc(gwf_test, head_filerecord='k_test.hds', budget_filerecord='k_test.cbc',
             saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')])

sim_test.write_simulation()
success_test, _ = sim_test.run_simulation(silent=True)

# --- Analyze results ---
print(f"{'='*55}")
print(f"  K = {k_test} m/day — Results")
print(f"{'='*55}")

if success_test:
    hds_test = flopy.utils.HeadFile(
        os.path.join(MODEL_WS, 'k_sensitivity', 'k_test.hds')).get_data()[0].flatten()
    
    active = idomain[0].flatten() > 0
    valid_test = active  # Newton solver always returns finite heads for active cells
    
    n_active_test = active.sum()
    
    if valid_test.sum() > 0:
        h_valid = hds_test[valid_test]
        above_surface = np.sum(h_valid > model_top[valid_test])
        # Newton solver never produces HDRY; instead, heads drop near/below aquifer bottom
        below_bottom = np.sum(h_valid < model_bottom[valid_test] + 0.5)
        print(f"  Converged:        Yes")
        print(f"  Head range:       {h_valid.min():.1f} to {h_valid.max():.1f} m a.s.l.")
        print(f"  Cells near/below aquifer bottom: {below_bottom} ({100*below_bottom/n_active_test:.1f}%)")
        print(f"  Cells above surface: {above_surface}")
        
        # Verdict
        if below_bottom > n_active_test * 0.1:
            print(f"\n  Verdict: Heads near/below aquifer bottom — K is likely TOO HIGH")
            print(f"  (aquifer drains faster than it is recharged)")
        elif above_surface > n_active_test * 0.05:
            print(f"\n  Verdict: Heads exceed land surface — K is likely TOO LOW")
            print(f"  (water backs up because it cannot flow out fast enough)")
        else:
            print(f"\n  Verdict: Physically reasonable simulation")
    else:
        print(f"  No active cells — check model setup!")
else:
    print(f"  Simulation FAILED to converge with K = {k_test} m/day")

> **🤔 Reflection: K Sensitivity**
>
> After testing several K values, consider:
> - Which K values produce a model where the aquifer neither drains nor fills up unrealistically?
> - Why does K matter so much for steady-state heads, even though it doesn't directly control how much water enters the model?
> - How does this exercise motivate the calibration work in Notebook 5?
>
> <details>
> <summary>Hint</summary>
>
> In a steady-state model, K controls the **gradient** needed to transmit a given flux. Low K requires steep gradients (high heads), while high K allows flat gradients (heads close to outflow level). The "correct" K is the one that produces gradients matching observed water table elevations.
> </details>

In [ ]:
# Checkpoint: K sensitivity
create_multiple_choice('task04_checkpoint_k_sensitivity')

### 7.6 - ✏️ Checkpoint 5: Model Calibration

**Reflection Question:** What observations would you compare to simulated values to assess model quality?

In [ ]:
# Checkpoint 5: What observations would you compare to simulated values?
check_task_with_solution('task04_checkpoint_5')

---

## 8 - Summary

In [ ]:
# Dynamic model summary
if success:
    print("=" * 60)
    print("MODEL SUMMARY")
    print("=" * 60)
    print(f"  Grid:              {ncpl} cells (DISV, 1 layer)")
    print(f"  Active cells:      {n_active}")
    print(f"  Model area:        {boundary_gdf.geometry.area.sum() / 1e6:.1f} km²")
    print(f"  Avg thickness:     {aquifer_thickness.mean():.1f} m")
    print(f"  K (uniform):       {k_uniform} m/day")
    print(f"  Transmissivity:    {k_uniform * aquifer_thickness.mean():.0f} m²/day")
    print(f"  Recharge:          {annual_recharge_mm} mm/year ({recharge_m_per_day:.2e} m/day)")
    lateral_total = sum(w[1] for w in wel_spd if w[1] > 0)
    pump_total = abs(sum(w[1] for w in pumping_spd)) if len(pumping_spd) > 0 else 0
    print(f"  Lateral inflow:    {lateral_total:.0f} m³/day (WEL)")
    print(f"  Pumping extraction:{pump_total:.0f} m³/day (WEL)")
    print(f"  River cells:       {len(riv_spd)} (RIV)")
    print(f"  CHD outflow head:  {chd_head_outflow:.1f} m a.s.l.")
    print(f"  Solver:            Newton-Raphson (UNDER_RELAXATION)")
    print(f"  Status:            {'Converged' if success else 'FAILED'}")
    print("=" * 60)

## What You're Taking Forward

| For Notebook 5 (Calibration) | Value/Insight |
|------------------------------|---------------|
| Working MODFLOW 6 model | River-aligned DISV grid with complete BCs |
| Dominant flux | River-aquifer interaction (RIV package) |
| Initial K estimate | 20 m/day (uniform, needs zone-based refinement) |
| Recharge | ~110 mm/year (3.0×10⁻⁴ m/day) |
| Pumping wells | Medium concessions ~1,640 m³/day (from Notebook 2 analysis) |
| Key calibration targets | K distribution (uniform → zoned), recharge rate, leakage coefficients |

**Key takeaways:**
- MODFLOW 6 with Newton-Raphson solver handles unconfined flow robustly
- **River-aligned grids** ensure accurate river-aquifer exchange calculations  
- DISV grids provide flexibility for local refinement in case study work
- The model is **uncalibrated** - do not use for predictions yet

**The big picture:** We have a working numerical model that formalizes your perceptual understanding. The next step is to optimize parameters against observed groundwater levels.

---

## Next Steps

**Continue to:** [5_calibration.ipynb](5_calibration.ipynb) – Calibrate model parameters against observed heads

In [ ]:
print(f"Model saved to: {MODEL_WS}")
print("This simulation will be the starting point for calibration in Notebook 5.")

---

**Navigation:** [< Notebook 3: MODFLOW Fundamentals](3_modflow_fundamentals.ipynb) | [Notebook 5: Calibration >](5_calibration.ipynb)

---

## 9 - Optional: Additional Details

### 9.1 - DISV Cell Numbering

DISV grids use 1D cell indexing (0 to ncpl-1). The `cell2d` array defines cell ID, center coordinates, and vertex connectivity.

In [ ]:
print("Example cell2d entries (first 3 cells):")
for i, cell in enumerate(gridprops['cell2d'][:3]):
    print(f"  Cell {cell[0]}: center=({cell[1]:.1f}, {cell[2]:.1f}), {cell[3]} vertices")

### 9.2 - Export Grid to GeoPackage

In [ ]:
if success:
    # Use masked heads for export (replace dry cells with NaN)
    head_export = np.where(
        (np.abs(head_layer1) > hdry_threshold) | (idomain[0].flatten() <= 0),
        np.nan,
        head_layer1
    )
    
    output_gpkg = os.path.join(MODEL_WS, 'model_results.gpkg')
    export_grid_to_geopackage(
        modelgrid, head_export, output_path=output_gpkg,
        layer_name='simulated_heads', array_name='head_m'
    )
    print(f"Results exported to: {output_gpkg}")

## References

[\[1\]](#2---Model-Grid---DISV) FloPy Documentation: Unstructured Grids and DISV. [https://flopy.readthedocs.io/](https://flopy.readthedocs.io/) (accessed 2026)

[\[2\]](#5---Boundary-Conditions) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.

[\[3\]](#5---Boundary-Conditions) MODFLOW 6 User Guide: River Package (RIV). U.S. Geological Survey. [https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model](https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model) (accessed 2026)

[\[4\]](#3---Model-Geometry) Federal Office of Topography swisstopo, swissALTI3D height model (2m resolution). [https://www.swisstopo.admin.ch/en/height-model-swissalti3d](https://www.swisstopo.admin.ch/en/height-model-swissalti3d) (accessed 2026)

[\[5\]](#3---Model-Geometry) GIS-browser of the canton of Zurich: Groundwater thickness data (GS_GW_MAECHTIGKEIT_L). [https://www.gis.zh.ch/](https://www.gis.zh.ch/) (accessed 2026)

[\[6\]](#2---Model-Grid---DISV) MODFLOW 6 Description of Input and Output: DISV Package. Langevin, C.D., et al. (2022). U.S. Geological Survey Techniques and Methods 6-A62.

[\[7\]](#6---Build-and-Run-Model) Hughes, J.D., Langevin, C.D., Banta, E.R. (2017): Documentation for the MODFLOW 6 framework. U.S. Geological Survey Techniques and Methods 6-A57, 42 p. (Newton solver guidance)